# NER — Named Entity Recognition

Runs NER across the vault using both backends (Ollama and Claude) and compares results
against the ground truth in `data/ground_truth.yaml`.

In [1]:
import sys
sys.path.insert(0, '../src')

from glob import glob
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

from notemine.config import load_config
from notemine.frontmatter_io import load_note
from notemine.ground_truth import load_ground_truth
from notemine.tasks import run_ner

load_dotenv('../.env')
config = load_config('../config.toml')
gt = load_ground_truth('../' + config['vault']['ground_truth'].lstrip('./'))

vault = '../' + config['vault']['path'].lstrip('./')
config['vault']['prompts_dir'] = '../' + config['vault']['prompts_dir'].lstrip('./')

notes = sorted(
    p for p in glob(f'{vault}/**/*.md', recursive=True)
    if not Path(p).name.startswith('index_')
    and not Path(p).suffix == '.yaml'
)
print(f'{len(notes)} notes found')

20 notes found


## Run NER with both backends

This cell runs NER on every note with both Ollama and Claude.
Errors are caught per-note so one failure doesn't stop the loop.

In [2]:
# BACKENDS = ['ollama', 'claude'] # TODO: add back Claude when we have results
BACKENDS = ['ollama']

results = []
for path in notes:
    name = Path(path).name
    post = load_note(path)
    if len(post.content.strip()) < 50:
        continue

    gt_entry = gt.get(name, {})
    gt_entities = [(e['entity'], e['type']) for e in gt_entry.get('entities', [])]

    row = {'file': name, 'lang': gt_entry.get('lang', '?'), 'ground_truth': gt_entities}

    for backend in BACKENDS:
        try:
            row[backend] = [(e[0], e[1]) for e in run_ner(backend, post.content, config)]
        except Exception as e:
            row[backend] = f'ERROR: {e}'

    results.append(row)
    print(f'  {name}')

print(f'\nDone — {len(results)} notes processed')

  long_ai_gezondheidszorg_oncologie_onderzoek_nl.md
  long_ai_healthcare_oncology_research_en.md
  long_interview_climate_scientist_sea_level_en.md
  long_interview_klimaatwetenschapper_zeespiegel_nl.md
  long_opinie_eu_ai_act_digitaal_beleid_nl.md
  long_opinion_eu_ai_act_digital_policy_en.md
  medium_climate_blog_paris_agreement_en.md
  medium_geschiedenis_gouden_eeuw_nl.md
  medium_history_dutch_golden_age_en.md
  medium_klimaat_blog_parijs_akkoord_nl.md
  medium_reisgids_amsterdam_nl.md
  medium_travel_guide_amsterdam_en.md
  short_recensie_samsung_galaxy_s25_ultra_nl.md
  short_recept_stamppot_nl.md
  short_recipe_dutch_stamppot_en.md
  short_review_samsung_galaxy_s25_ultra_en.md
  short_sport_nieuws_ajax_champions_league_nl.md
  short_sports_news_ajax_champions_league_en.md
  short_tech_news_openai_announcement_en.md
  short_tech_nieuws_openai_aankondiging_nl.md

Done — 20 notes processed


## Summary comparison

Shows how many ground-truth entities each backend matched, plus how many extra it added.

In [3]:
def ner_stats(gt_ents, pred):
    if isinstance(pred, str):
        return pred
    gt_set = set(gt_ents)
    pred_set = set(pred)
    matched = len(gt_set & pred_set)
    missed = len(gt_set - pred_set)
    extra = len(pred_set - gt_set)
    pct = int(100 * matched / len(gt_set)) if gt_set else 0
    return f'{matched}/{len(gt_set)} ({pct}%)  +{extra} extra'

rows = []
for r in results:
    rows.append({
        'file': r['file'],
        'lang': r['lang'],
        'gt_count': len(r['ground_truth']),
        'ollama': ner_stats(r['ground_truth'], r['ollama']),
        # 'claude': ner_stats(r['ground_truth'], r['claude']), # TODO: add back when we have Claude results
    })

pd.set_option('display.max_colwidth', 300)
pd.DataFrame(rows)

,file,lang,gt_count,ollama
0,long_ai_gezondheidszorg_oncologie_onderzoek_nl.md,nl,48,ERROR: Expecting value: line 1 column 1 (char 0)
1,long_ai_healthcare_oncology_research_en.md,en,48,27/48 (56%) +15 extra
2,long_interview_climate_scientist_sea_level_en.md,en,45,ERROR: Expecting value: line 1 column 1 (char 0)
3,long_interview_klimaatwetenschapper_zeespiegel_nl.md,nl,45,ERROR: Expecting value: line 1 column 1 (char 0)
4,long_opinie_eu_ai_act_digitaal_beleid_nl.md,nl,47,ERROR: 0
5,long_opinion_eu_ai_act_digital_policy_en.md,en,47,30/47 (63%) +18 extra
6,medium_climate_blog_paris_agreement_en.md,en,41,ERROR: Expecting value: line 1 column 1 (char 0)
7,medium_geschiedenis_gouden_eeuw_nl.md,nl,36,8/36 (22%) +25 extra
8,medium_history_dutch_golden_age_en.md,en,36,18/36 (50%) +29 extra
9,medium_klimaat_blog_parijs_akkoord_nl.md,nl,41,24/41 (58%) +19 extra


## Inspect a single note

Change `INSPECT` to any filename to see the full entity lists side by side.

In [ ]:
INSPECT = Path(notes[15]).name  # change to any note filename

row = next((r for r in results if r['file'] == INSPECT), None)
if row is None:
    print(f'{INSPECT} not found in results')
else:
    gt_set = set(row['ground_truth'])
    ol = set(row['ollama']) if not isinstance(row['ollama'], str) else set()
    cl = set(row['claude']) if not isinstance(row['claude'], str) else set()

    all_ents = sorted(gt_set | ol | cl)
    rows = []
    for ent in all_ents:
        rows.append({
            'entity': ent[0],
            'type': ent[1],
            'ground_truth': '✓' if ent in gt_set else '',
            'ollama': '✓' if ent in ol else '',
            'claude': '✓' if ent in cl else '',
        })

    print(f'=== {INSPECT} ===')
    pd.set_option('display.max_colwidth', 50)
    display(pd.DataFrame(rows))

## (Optional) Save results to frontmatter

Uncomment and run to write results to note files under `notemine.ner.ollama` / `notemine.ner.claude`.
Run `python main.py reset` to undo.

In [ ]:
# from notemine.frontmatter_io import save_note
#
# for path in notes:
#     name = Path(path).name
#     row = next((r for r in results if r['file'] == name), None)
#     if row is None:
#         continue
#     post = load_note(path)
#     notemine = post.metadata.get('notemine', {})
#     ner_data = notemine.get('ner', {})
#     if not isinstance(row.get('ollama'), str):
#         ner_data['ollama'] = [list(e) for e in row['ollama']]
#     if not isinstance(row.get('claude'), str):
#         ner_data['claude'] = [list(e) for e in row['claude']]
#     notemine['ner'] = ner_data
#     post.metadata['notemine'] = notemine
#     save_note(path, post)
# print('Saved.')